In [ ]:
import os
import json
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from huggingface_hub import login
from sklearn.metrics import accuracy_score, precision_score, recall_score, ndcg_score
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader

login("your_huggingface_auth_token_here")

# Disable wandb logging
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
def load_law_corpus(corpus_path):
    print(f"📚 Loading law corpus from {corpus_path}...")
    with open(corpus_path, 'r', encoding='utf-8') as f:
        corpus_data = json.load(f)
    
    corpus = []
    
    for law in corpus_data:
        law_id = law.get("id", "unknown_law")
        
        if "articles" in law:
            for article in law["articles"]:
                article_id = article.get("id", "unknown_article")
                content = article.get("text", "")
                
                doc_id = f"{law_id}::{article_id}"
                
                if content.strip():
                    corpus.append({
                        "id": doc_id,
                        "text": content.strip(),
                        "law_id": str(law_id),
                        "article_id": str(article_id)
                    })
        else:
            print(f"⚠️ Warning: Law {law_id} has no articles field")
    
    print(f"✅ Loaded {len(corpus)} law articles")
    
    if corpus:
        print(f"📋 Sample document: {corpus[0]['id']}")
        print(f"📄 Sample text: {corpus[0]['text'][:100]}...")
    
    return corpus

In [ ]:
def load_training_queries(queries_path, corpus):
    """
    Load training queries từ alqiac_2025_train_preview.json và tạo qrels
    """
    print(f"❓ Loading training queries from {queries_path}...")
    with open(queries_path, 'r', encoding='utf-8') as f:
        queries_data = json.load(f)
    
    queries = []
    qrels = []
    
    article_to_doc_id = {}
    for doc in corpus:
        if "::" in doc['id']:
            law_id_from_doc, article_id_from_doc = doc['id'].split("::", 1)
            key = (law_id_from_doc, article_id_from_doc)
            article_to_doc_id[key] = doc['id']
    
    print(f"🗂️ Created mapping for {len(article_to_doc_id)} articles")
    
    for item in queries_data:
        query_id = item.get('question_id', item.get('id', 'unknown_query'))
        query_text = item.get('text', item.get('question', ''))
        
        queries.append({
            "id": query_id,
            "text": query_text
        })
        
        if 'relevant_articles' in item:
            for rel_article in item['relevant_articles']:
                law_id = rel_article['law_id']
                article_id = rel_article['article_id']
                
                key = (law_id, article_id)
                if key in article_to_doc_id:
                    doc_id = article_to_doc_id[key]
                    qrels.append({
                        "query_id": query_id,
                        "doc_id": doc_id,
                        "score": 1
                    })
                else:
                    print(f"⚠️ Warning: Article not found: {law_id} - {article_id}")
    
    print(f"✅ Loaded {len(queries)} training queries with {len(qrels)} relevance judgments")
    return queries, qrels

In [ ]:
def load_test_queries(test_path, corpus):
    """
    Load test queries từ alqac25_private_test_task2.json
    """
    print(f"🧪 Loading test queries from {test_path}...")
    with open(test_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
    
    test_queries = []
    test_qrels = []
    
    article_to_doc_id = {}
    for doc in corpus:
        if "::" in doc['id']:
            law_id_from_doc, article_id_from_doc = doc['id'].split("::", 1)
            key = (law_id_from_doc, article_id_from_doc)
            article_to_doc_id[key] = doc['id']
    
    for item in test_data:
        query_id = item.get('question_id', item.get('id', 'unknown_test_query'))
        query_text = item.get('text', item.get('question', ''))
        
        test_queries.append({
            "id": query_id,
            "text": query_text
        })
        
        if 'relevant_articles' in item:
            for rel_article in item['relevant_articles']:
                law_id = rel_article['law_id']
                article_id = rel_article['article_id']
                
                key = (law_id, article_id)
                if key in article_to_doc_id:
                    doc_id = article_to_doc_id[key]
                    test_qrels.append({
                        "query_id": query_id,
                        "doc_id": doc_id,
                        "score": 1
                    })
    
    print(f"✅ Loaded {len(test_queries)} test queries with {len(test_qrels)} relevance judgments")
    return test_queries, test_qrels

In [ ]:
def create_data_splits(queries, n_validation=100):
    """
    Chia data theo yêu cầu:
    - Validation: 100 samples đầu tiên
    - Training: các samples còn lại
    """
    n_queries = len(queries)
    n_validation = min(n_validation, n_queries // 2)
    
    if n_queries <= 100:
        print(f"⚠️ Warning: Total queries ({n_queries}) <= 100. Using 50% for validation.")
        n_validation = n_queries // 2
    
    validation_indices = list(range(n_validation))
    train_indices = list(range(n_validation, n_queries))
    
    splits = {
        "train": [queries[i]["id"] for i in train_indices],
        "validation": [queries[i]["id"] for i in validation_indices]
    }
    
    print(f"📊 DATA SPLIT:")
    print(f"   🏋️ Training: {len(train_indices)} samples")
    print(f"   ✅ Validation: {len(validation_indices)} samples (first {n_validation})")
    print(f"   📈 Hard negative mining will use {len(train_indices)} training samples")
    
    return splits

In [ ]:
def save_processed_data(corpus, queries, qrels, test_queries, test_qrels, splits, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    with open(os.path.join(output_dir, "corpus.jsonl"), "w", encoding="utf-8") as f:
        for doc in corpus:
            f.write(json.dumps(doc, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "training_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "test_queries.jsonl"), "w", encoding="utf-8") as f:
        for query in test_queries:
            f.write(json.dumps(query, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "training_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "test_qrels.jsonl"), "w", encoding="utf-8") as f:
        for qrel in test_qrels:
            f.write(json.dumps(qrel, ensure_ascii=False) + "\n")
    
    with open(os.path.join(output_dir, "data_splits.json"), "w", encoding="utf-8") as f:
        json.dump(splits, f, ensure_ascii=False, indent=2)
    
    print(f"💾 Saved processed data to {output_dir}")

In [ ]:
def generate_hard_negatives(corpus, queries, qrels, splits, n_hard_negatives=5, model_name="FacebookAI/xlm-roberta-large"):
    print(f"⚡ Generating hard negatives using {model_name}...")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    
    corpus_texts = [item["text"] for item in corpus]
    corpus_ids = [item["id"] for item in corpus]
    print(f"🔢 Encoding {len(corpus_texts)} documents...")
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True)
    
    query_to_positives = {}
    for qrel in qrels:
        if qrel["query_id"] not in query_to_positives:
            query_to_positives[qrel["query_id"]] = []
        query_to_positives[qrel["query_id"]].append(qrel["doc_id"])
    
    qid_to_query = {item["id"]: item["text"] for item in queries}
    
    train_examples = []
    train_queries = splits["train"]
    
    print(f"🔍 Mining hard negatives for {len(train_queries)} training queries...")
    
    for query_id in tqdm(train_queries, desc="Hard negative mining"):
        if query_id not in query_to_positives or query_id not in qid_to_query:
            continue
            
        query_text = qid_to_query[query_id]
        positive_ids = query_to_positives[query_id]
        
        positive_texts = []
        for doc_id in positive_ids:
            if doc_id in corpus_ids:
                idx = corpus_ids.index(doc_id)
                positive_texts.append(corpus_texts[idx])
        
        if not positive_texts:
            continue
        
        query_embedding = model.encode(query_text, convert_to_tensor=True)
        
        cos_scores = torch.nn.functional.cosine_similarity(query_embedding, corpus_embeddings)
        
        hard_negative_indices = []
        cos_scores_np = cos_scores.cpu().numpy()
        sorted_indices = np.argsort(-cos_scores_np)
        
        for idx in sorted_indices:
            doc_id = corpus_ids[idx]
            if doc_id not in positive_ids:
                hard_negative_indices.append(idx)
                if len(hard_negative_indices) >= n_hard_negatives:
                    break
        
        hard_negative_texts = [corpus_texts[idx] for idx in hard_negative_indices]
        
        for pos_text in positive_texts:
            train_examples.append(InputExample(
                texts=[query_text, pos_text] + hard_negative_texts[:3]
            ))
    
    print(f"🎲 Adding random negatives for diversity...")
    for query_id in train_queries:
        if query_id not in query_to_positives or query_id not in qid_to_query:
            continue
            
        query_text = qid_to_query[query_id]
        positive_ids = query_to_positives[query_id]
        
        for pos_id in positive_ids:
            if pos_id not in corpus_ids:
                continue
                
            pos_idx = corpus_ids.index(pos_id)
            pos_text = corpus_texts[pos_idx]
            
            random_neg_indices = []
            while len(random_neg_indices) < 3:
                idx = random.randint(0, len(corpus_texts) - 1)
                if corpus_ids[idx] not in positive_ids and idx not in random_neg_indices:
                    random_neg_indices.append(idx)
            
            random_neg_texts = [corpus_texts[idx] for idx in random_neg_indices]
            
            train_examples.append(InputExample(
                texts=[query_text, pos_text] + random_neg_texts
            ))
    
    print(f"✅ Generated {len(train_examples)} training examples with hard negatives")
    return train_examples

In [ ]:
def setup_validation_evaluator(corpus, queries, qrels, splits):
    validation_queries = {q["id"]: q["text"] for q in queries if q["id"] in splits["validation"]}
    validation_relevant_docs = {}
    
    for qrel in qrels:
        if qrel["query_id"] in validation_queries:
            if qrel["query_id"] not in validation_relevant_docs:
                validation_relevant_docs[qrel["query_id"]] = []
            validation_relevant_docs[qrel["query_id"]].append(qrel["doc_id"])
    
    corpus_dict = {doc["id"]: doc["text"] for doc in corpus}
    
    print(f"✅ Setup validation evaluator: {len(validation_queries)} queries")
    
    return evaluation.InformationRetrievalEvaluator(
        validation_queries, corpus_dict, validation_relevant_docs, name='validation-eval'
    )

In [ ]:
def fine_tune_model(train_examples, validation_evaluator, output_dir, n_epochs=1, model_name="FacebookAI/xlm-roberta-large"):
    print(f"🤖 Loading model: {model_name}")
    model = SentenceTransformer(model_name, trust_remote_code=True)
    
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
    
    train_loss = losses.MultipleNegativesRankingLoss(model)
    
    print(f"🏋️ Starting training...")
    print(f"   📊 Training examples: {len(train_examples)}")
    print(f"   🔄 Epochs: {n_epochs}")
    print(f"   📍 Model will be saved to: {os.path.join(output_dir, 'model')}")
    
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        evaluator=validation_evaluator,
        epochs=n_epochs,
        evaluation_steps=100,
        warmup_steps=100,
        optimizer_params={'lr': 1.5e-5},
        output_path=os.path.join(output_dir, "model"),
        save_best_model=True
    )
    
    print(f"✅ Training completed! Model saved.")
    return model

In [ ]:
def evaluate_on_test_set(model, corpus, test_queries, test_qrels, output_dir):
    print(f"🧪 Evaluating on test benchmark...")
    
    test_queries_dict = {q["id"]: q["text"] for q in test_queries}
    test_relevant_docs = {}
    
    for qrel in test_qrels:
        if qrel["query_id"] not in test_relevant_docs:
            test_relevant_docs[qrel["query_id"]] = []
        test_relevant_docs[qrel["query_id"]].append(qrel["doc_id"])
    
    corpus_dict = {doc["id"]: doc["text"] for doc in corpus}
    
    test_evaluator = evaluation.InformationRetrievalEvaluator(
        test_queries_dict, corpus_dict, test_relevant_docs, name='test-benchmark'
    )
    
    test_results = test_evaluator(model)
    
    with open(os.path.join(output_dir, "test_benchmark_results.json"), "w", encoding="utf-8") as f:
        json.dump(test_results, f, indent=4, ensure_ascii=False)
    
    print(f"📊 TEST BENCHMARK RESULTS:")
    for metric, value in test_results.items():
        print(f"   {metric}: {value:.4f}")
    
    return test_results

In [ ]:
def main_pipeline(corpus_path, queries_path, test_path, output_dir, model_name="FacebookAI/xlm-roberta-large", n_epochs=1):
    print("🚀 ALQAC 2025 IR FINE-TUNING PIPELINE")
    print("=" * 60)
    print(f"📚 Law Corpus: {corpus_path}")
    print(f"❓ Training Queries: {queries_path}")
    print(f"🧪 Test Benchmark: {test_path}")
    print(f"🤖 Model: {model_name}")
    print(f"🔄 Epochs: {n_epochs}")
    print("=" * 60)
    

    print("\n📋 STEP 1: Loading data...")
    corpus = load_law_corpus(corpus_path)
    queries, qrels = load_training_queries(queries_path, corpus)
    test_queries, test_qrels = load_test_queries(test_path, corpus)
    
    print("\n📊 STEP 2: Creating data splits...")
    splits = create_data_splits(queries, n_validation=82)
    
    print("\n💾 STEP 3: Saving processed data...")
    save_processed_data(corpus, queries, qrels, test_queries, test_qrels, splits, output_dir)
    
    print("\n⚡ STEP 4: Generating hard negatives...")
    train_examples = generate_hard_negatives(corpus, queries, qrels, splits, model_name=model_name)
    
    print("\n✅ STEP 5: Setting up validation evaluator...")
    validation_evaluator = setup_validation_evaluator(corpus, queries, qrels, splits)
    
    print("\n🏋️ STEP 6: Fine-tuning model...")
    model = fine_tune_model(train_examples, validation_evaluator, output_dir, n_epochs, model_name)
    
    print("\n🧪 STEP 7: Test benchmark evaluation...")
    test_results = evaluate_on_test_set(model, corpus, test_queries, test_qrels, output_dir)
    
    print("\n🎉 PIPELINE COMPLETED SUCCESSFULLY!")
    print(f"📁 All results saved to: {output_dir}")
    print("=" * 60)
    
    return {
        "model": model,
        "test_results": test_results,
        "data_info": {
            "corpus_size": len(corpus),
            "train_queries": len(splits["train"]),
            "validation_queries": len(splits["validation"]),
            "test_queries": len(test_queries),
            "training_examples": len(train_examples)
        }
    }

In [ ]:
corpus_path = "./ALQAC_2025/alqac25_law.json"
queries_path = "./ALQAC_2025/alqac25_train.json"
test_path = "./alqac-2022-2025/alqac25_private_test_task2.json"
output_dir = "./working/"


model_name = "./models/ViLegalBERT"
n_epochs = 1

results = main_pipeline(
    corpus_path=corpus_path,
    queries_path=queries_path,
    test_path=test_path,
    output_dir=output_dir,
    model_name=model_name,
    n_epochs=n_epochs
)

print(f"\n📈 FINAL SUMMARY:")
print(f"✅ Model trained and saved")
print(f"📊 Data processed: {results['data_info']}")
print(f"🧪 Test results available in: {output_dir}/test_benchmark_results.json")